In [1]:
import os, sys
os.environ['NUMBA_THREADING_LAYER'] = 'workqueue'  # PySDM & PyMPDATA don't work with TBB; OpenMP has extra dependencies on macOS
if 'google.colab' in sys.modules:
    !pip --quiet install open-atmos-jupyter-utils
    from open_atmos_jupyter_utils import pip_install_on_colab
    pip_install_on_colab('PySDM-examples', 'PySDM')

In [2]:
import subprocess
import shutil
SUBPROCESS_ENV = os.environ.copy()

if 'google.colab' in sys.modules:
    !apt-get install -qq ghostscript
    !wget -nv "https://paraview.org/paraview-downloads/download.php?submit=Download&version=v5.13&type=binary&os=Linux&downloadFile=ParaView-5.13.1-egl-MPI-Linux-Python3.10-x86_64.tar.gz" -O paraview.tar.gz
    !tar xzf paraview.tar.gz
    SUBPROCESS_ENV['PATH'] += ':' + subprocess.check_output(['bash', '-c', "echo `pwd`/`dirname ParaView*/bin/pvpython`"], text=True)[:-1]
    
    # check if Ghostscript's ps2pdf works
    assert subprocess.check_call(['type', 'ps2pdf'], shell=True) == 0

# Find pvpython location
pvpython_path = shutil.which('pvpython')
if pvpython_path is None:
    # Try with bash to load bashrc
    result = subprocess.run(['bash', '-ic', 'which pvpython'], capture_output=True, text=True)
    pvpython_path = result.stdout.strip()

if not pvpython_path:
    raise FileNotFoundError("pvpython not found in PATH")

# check if Paraview's pvpython works
assert subprocess.check_call([pvpython_path, '--version']) == 0
assert subprocess.check_call([pvpython_path, '-c', 'import paraview']) == 0

paraview version 5.13.3


In [3]:
from PySDM_examples.Arabas_et_al_2015 import Settings, SpinUp
from PySDM_examples.utils.kinematic_2d import Simulation, Storage
from PySDM.exporters import VTKExporter
from PySDM_examples.utils import ProgBarController
from PySDM.physics import si
from PySDM import products as PySDM_products
import PySDM_examples
import glob
import platform
import pathlib

In [4]:
products = [
    PySDM_products.EffectiveRadius(unit='um'),
    PySDM_products.FlowVelocityComponent(component = 0, name = 'cx'),
    PySDM_products.FlowVelocityComponent(component = 1, name = 'cy')
]

##### 1. run a simulations saving output to VTK files

In [ ]:
settings = Settings()
settings.simulation_time = 100 * si.minute
storage = Storage()
simulation = Simulation(settings, storage, SpinUp=SpinUp)
simulation.reinit(products)

vtk_exporter = VTKExporter(path='.')    

simulation.run(ProgBarController("progress:"), vtk_exporter=vtk_exporter)
vtk_exporter.write_pvd()

nvrtc not found at /usr/local/cuda/lib64/libnvrtc.so
Loading libnvrtc failed. 
nvrtc not found at /usr/local/cuda/lib64/libnvrtc.so
Loading libnvrtc failed. 
cuMemAlloc() failed with Error code: 2
Error Name: CUDA_ERROR_OUT_OF_MEMORY
Error Description: out of memory
nvrtc not found at /usr/local/cuda/lib64/libnvrtc.so
Loading libnvrtc failed. 
cuMemAlloc() failed with Error code: 2
Error Name: CUDA_ERROR_OUT_OF_MEMORY
Error Description: out of memory
nvrtc not found at /usr/local/cuda/lib64/libnvrtc.so
Loading libnvrtc failed. 
cuMemAlloc() failed with Error code: 2
Error Name: CUDA_ERROR_OUT_OF_MEMORY
Error Description: out of memory
nvrtc not found at /usr/local/cuda/lib64/libnvrtc.so
Loading libnvrtc failed. 
cuMemAlloc() failed with Error code: 2
Error Name: CUDA_ERROR_OUT_OF_MEMORY
Error Description: out of memory
nvrtc not found at /usr/local/cuda/lib64/libnvrtc.so
Loading libnvrtc failed. 
cuMemAlloc() failed with Error code: 2
Error Name: CUDA_ERROR_OUT_OF_MEMORY
Error Descript

SystemError: An internal error happend

#### 2. execute ``PySDM_examples/utils/pvanim.py`` script using `pvpython`

In [ ]:
pvanim = pathlib.Path(PySDM_examples.__file__).parent / "utils" / "pvanim.py"
result = subprocess.run([pvpython_path, str(pvanim), '--help'], check=True, env=SUBPROCESS_ENV)

Launcher options:
  --print       Print modified environment.
  --system-mpi  Use MPI implementation available on the system.
  --mesa        Use Mesa GL for rendering.
  --backend <backend>  Specify mesa backend.

Available backends:
    llvmpipe
    swr

pvpython options:


usage: pvanim.py [-h] [--mode {light,dark}]
                 [--multiplicity_preset MULTIPLICITY_PRESET]
                 [--multiplicity_logscale]
                 [--effectiveradius_preset EFFECTIVERADIUS_PRESET]
                 [--effectiveradius_logscale]
                 [--effectiveradius_nan_color EFFECTIVERADIUS_NAN_COLOR EFFECTIVERADIUS_NAN_COLOR EFFECTIVERADIUS_NAN_COLOR]
                 [--sd_products_opacity SD_PRODUCTS_OPACITY]
                 [--calculator1_opacity CALCULATOR1_OPACITY]
                 [--sd_attributes_opacity SD_ATTRIBUTES_OPACITY]
                 [--animation_size ANIMATION_SIZE ANIMATION_SIZE]
                 [--animationframename ANIMATIONFRAMENAME]
                 [--animationname ANIMATIONNAME] [--framerate FRAMERATE]
                 product_path attributes_path output_path

positional arguments:
  product_path          path to pvd products file
  attributes_path       path to pvd attributes file
  output_path           path where to write ou

In [ ]:
product = pathlib.Path("./output/sd_products.pvd").absolute()
attributes = pathlib.Path("./output/sd_attributes.pvd").absolute()

try:
    for mode in ('light', 'dark'):
        result = subprocess.run(
            [
                pvpython_path,
                "--force-offscreen-rendering",
                str(pvanim),
                str(product),
                str(attributes),
                str(pathlib.Path('./output').absolute()),
                "--animationname", "docs_intro_animation.ogv",
                "--mode", mode,
            ] + (["--animationframename", "last_animation_frame.pdf"] if mode == 'light' else []),
            check=platform.system() != "Windows",
            capture_output=True,
            text=True,
            env=SUBPROCESS_ENV,
        )
except subprocess.CalledProcessError as e:
    print(e.stderr)
    assert False

#### 3. reduce file size for generated pdf files

In [ ]:
if platform.system() != 'Windows':
    for file in glob.glob('output/anim_frame_*.pdf'):
        subprocess.run(['ps2pdf', file, file+'_'], capture_output=True, check=True)
        subprocess.run(['mv', file+'_', file], check=True)